In [1]:
!pip install pandas openpyxl numpy

# Lihat Preview Dataset

In [11]:
import pandas as pd

nama_file_csv = 'dataset_properti_raw.csv'

try:
    df = pd.read_csv(nama_file_csv)
    
    display(df.tail())

except FileNotFoundError:
    print(f"Error: File '{nama_file_csv}' tidak ditemukan.")
    print("Pastikan file CSV sudah di-upload ke folder yang sama dengan notebook ini.")

,ad_id,title,price,lat,lon,Provinsi,Kota/kabupaten,Kecamatan,Luas_bangunan,Luas_tanah,Kamar_tidur,Kamar_Mandi,parameter-external_source_url,Tipe,Fasilitas,Lantai,description
7242,945160885,DIJUAL APARTEMEN PUNCAK DHARMAHUSADA TOWER B L...,3.550000e+08,-7.258,112.780,Jawa Timur,Surabaya Kota,Tambaksari,36.0,0.0,2,1,https://www.lamudi.co.id/properti/41032-73-8af...,apartemen,ac,7.0,DIJUAL Apartemen Puncak Dharmahusada Tower B ...
7243,939970814,Rumah murah d sidosermo Prapen Jemursari margo...,2.500000e+09,-7.271,112.748,Jawa Timur,Surabaya Kota,Gubeng,180.0,206.0,4,3,https://www.lamudi.co.id/properti/41032-73-65a...,rumah,NaN,NaN,"BANTING HARGA, MURAH Dijual Rumah Siap HuniLok..."
7244,941427709,Dijual Rumah PAKUWON CITY Sorento,1.900000e+09,-7.257,112.752,Jawa Timur,Surabaya Kota,Genteng,224.0,133.0,2,2,https://www.lamudi.co.id/properti/41032-73-8c6...,rumah,NaN,2.0,Dijual Rumah PAKUWON CITY Sorento Luas Tanah ...
7245,945160506,"Rumah Tengah Kota under 2M, Kompleks Perumahan...",1.950000e+09,-7.281,112.767,Jawa Timur,Surabaya Kota,Mulyorejo,125.0,82.0,3,3,https://www.lamudi.co.id/properti/41032-73-b21...,rumah,NaN,2.0,"Rumah Tengah Kota under 2M, Kompleks Perumahan..."
7246,945160497,‼️BARU GRESS MEWAH SPEK PREMIUM ‼️RUMAH SAN DI...,5.800000e+09,-7.279,112.806,Jawa Timur,Surabaya Kota,Sukolilo,310.0,200.0,5,4,https://www.lamudi.co.id/properti/41032-73-f4a...,rumah,NaN,2.0,BARU GRESS MEWAH SPEK PREMIUM RUMAH SAN DIEGO...


# Konversi ke Excel

In [9]:
nama_file_excel = 'dataset_properti_enriched.xlsx'

try:
    df.to_excel(nama_file_excel, index=False, engine='openpyxl')
    
except Exception as e:
    print(f"Terjadi kesalahan saat menyimpan file: {e}")

# Standardisasi Istilah Fasilitas

In [ ]:
synonym_map = {
    "garden": "Taman",
    "garasi": "Parkiran mobil",
    "carport": "Parkiran mobil",
    "swimmingpool": "Kolam renang",
    "gordyn": "Gorden",
    "pam": "Air",
    "waterheater": "Pemanas air",
    "refrigerator": "Kulkas",
    "stove": "Kompor",
    "fireextenguisher": "APAR (Alat Pemadam Api Ringan)",
    "ac": "Pendingin ruangan (AC)",
    "pendingin ruangan (ac)": "Pendingin ruangan (AC)",
    "keamanan": "Sistem Keamanan",
    "keamanan 24 jam": "Sistem Keamanan",
    "telephone": "Telepon"
}

def process_facilities(row):
    # Gabungkan kolom "Fasilitas", "Fasilitas_Indoor", dan "Karakteristik_bangunan"
    f1 = str(row['Fasilitas']) if pd.notna(row['Fasilitas']) else ""
    f2 = str(row['Fasilitas_Indoor']) if pd.notna(row['Fasilitas_Indoor']) else ""
    f3 = str(row['Karakteristik_bangunan']) if pd.notna(row['Karakteristik_bangunan']) else ""
    
    raw_combined = f"{f1},{f2},{f3}"
    items = [item.strip().lower() for item in raw_combined.split(',')]
    
    standardized_items = set()
    for item in items:
        if item == "" or item == "nan":
            continue
        mapped_value = synonym_map.get(item, item.title())
        standardized_items.add(mapped_value)
        
    return ", ".join(sorted(standardized_items))

# Eksekusi Standardisasi

In [16]:
df['Facilities'] = df.apply(process_facilities, axis=1)

df = df.drop(columns=['Fasilitas', 'Fasilitas_Indoor', 'Karakteristik_bangunan'])

df[['title', 'Facilities']].head(5)

,title,Facilities
0,"Rumah Baru 2 Lantai, Bagus dan Strategis di Wi...","Air, Listrik, Pagar Penuh, Parkiran mobil, Tan..."
1,Lantai 2‼️155 jt • 2 BR Termurah Apartemen Pun...,Sebagian Perabotan
2,Diamond Hill Citraland,"Parkiran mobil, Tanpa Perabotan"
3,"‼️BARU GRESS 2 UNIT‼️ RUMAH WISMA MUKTI, SEMAL...",Tanpa Perabotan
4,RUMAH MEWAH SIAP HUNI 2 LT DHARMAHUSADA SURABA...,"Listrik, Tanpa Perabotan"


# Simpan Updated Dataset

In [17]:
df.to_csv("pilot_testing_cleaned.csv", index=False)